# Intermediate Machine Learning

## 1. Missing Values

In [2]:
# imports
import pandas as pd
from sklearn.model_selection import train_test_split

# load the data
data = pd.read_csv('./datasets/melb_data.csv')

# select target
y = data.Price

# to keep things simple, we'll use only numerical predictors
melb_predictors = data.drop(['Price'], axis = 1)
X = melb_predictors.select_dtypes(exclude = ['object'])

# divide data into training and test subset
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.2, test_size = 0.8,
                                                    random_state = 0)

- define function to measure the quality of each approach
- define function score_dataset() to compare different approaches dealing with missing values
- this function reports the mean absolute error (MAE) fromv a random forest model

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# function for comparing different approaches
def score_dataset(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(n_estimators = 10, random_state = 0)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

### Approach 1 - Drop columns with missing values
- since we are working with both training and test sets, we are careful to drop the same columns in both dataframes

In [4]:
# get the names of columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

# drop columns in training and test data
reduced_X_train = X_train.drop(cols_with_missing, axis = 1)
reduced_X_test = X_test.drop(cols_with_missing, axis = 1)

print('MAE from Approach 1:')
print(score_dataset(reduced_X_train, reduced_X_test, y_train, y_test))

MAE from Approach 1:
205137.16743065784


### Approach 2 - Imputation
- next, we use SimpleImputer to replace missing values with the mean value along each column

In [5]:
from sklearn.impute import SimpleImputer

# imputation
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_test = pd.DataFrame(my_imputer.transform(X_test))

# impuation removed column names, put them back
imputed_X_train.columns = X_train.columns
imputed_X_test.columns = X_test.columns

print('MAE from Approach 2:')
print(score_dataset(imputed_X_train, imputed_X_test, y_train, y_test))

MAE from Approach 2:
198253.83823944526


... we see that Approach 2 has lower MAE than Approach 1, so Approach 2 performed better on this dataset

### Approach 3 - An extension to imputation
- next, we impute the missing values, while also keeping track of which values were imputed

In [6]:
# make a copy to avoid changing original data (when imputing)
X_train_plus = X_train.copy(deep = True)
X_test_plus = X_test.copy(deep = True)

# make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_test_plus[col + '_was_missing'] = X_test_plus[col].isnull()
    
# imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_test_plus = pd.DataFrame(my_imputer.transform(X_test_plus))

# imputation removed column names, put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_test_plus.columns = X_test_plus.columns

print('MAE from Aproach 3:')
print(score_dataset(imputed_X_train_plus, imputed_X_test_plus, y_train, y_test))

MAE from Aproach 3:
198005.93757363773


In [7]:
# shape of training data (num_rows, num_columns)
print(X_train.shape)

# number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(2716, 12)
Car               11
BuildingArea    1267
YearBuilt       1083
dtype: int64


### Conclusion
- as it is common, imputing missing values (in Approach 2 and Approach 3) yielded better results, relative to when we simply dropped columns with missing values (in Approach 1)

## 2. Categorical Values